In [28]:
import pandapipes as pp
from tespy.components import Compressor,SimpleHeatExchanger,CycleCloser, Valve, HeatExchanger,Condenser, Sink, Source
from tespy.connections import Connection
from tespy.networks import Network
import numpy as np
from tespy.tools import UserDefinedEquation
nw_Cooling_net = Network()
nw_Cooling_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
Cooling_net_compressor = Compressor("compresor")
Cooling_net_condenser = Condenser("condensador")
Cooling_net_valve = Valve("valvula_expansion")
Cooling_net_evaporator = HeatExchanger("evaporador")
Cooling_net_cc=CycleCloser('CycleCloser')
Cooling_net_source_consumer=Source("Source_Consumer ")
Cooling_net_sink_consumer=Sink("Sink_consumer")
Cooling_net_source_reseau=Source("Source_Reseau")
Cooling_net_sink_reseau=Sink("Sink_Reseau")
Cooling_net_c0=Connection(Cooling_net_valve, 'out1', Cooling_net_cc, 'in1', label='0')
Cooling_net_c1 = Connection(Cooling_net_cc, 'out1', Cooling_net_evaporator, 'in2', label='1')
Cooling_net_c2 = Connection(Cooling_net_evaporator, 'out2', Cooling_net_compressor, 'in1', label='2')
Cooling_net_c3 = Connection(Cooling_net_compressor, 'out1', Cooling_net_condenser, 'in1', label='3')
Cooling_net_c4 = Connection(Cooling_net_condenser, 'out1', Cooling_net_valve, 'in1', label='4')
Cooling_net_c5=Connection(Cooling_net_condenser, 'out2',Cooling_net_sink_consumer, 'in1', label='5')
Cooling_net_c6=Connection(Cooling_net_source_consumer, 'out1',Cooling_net_condenser , 'in2', label='6')
Cooling_net_c7=Connection(Cooling_net_source_reseau, 'out1',Cooling_net_evaporator , 'in1', label='7')
Cooling_net_c8=Connection(Cooling_net_evaporator, 'out1',Cooling_net_sink_reseau , 'in1', label='8')
nw_Cooling_net.add_conns(Cooling_net_c0,  Cooling_net_c1,  Cooling_net_c2,  Cooling_net_c3,  Cooling_net_c4 , Cooling_net_c5,  Cooling_net_c6, Cooling_net_c7, Cooling_net_c8)
def my_ude(ude):
    return ude.conns[0].calc_T_dew() +5-ude.conns[1].calc_T()
def my_ude_dependents(ude):
    c1, c2 = ude.conns
    return [c1.p,c1.h, c2.p,c2.h]
def my_ude_2(ude):
    return ude.conns[0].calc_T() +5-ude.conns[1].calc_T()
def my_ude_dependents_2(ude):
    c1, c2 = ude.conns
    return [c1.p,c1.h, c2.p,c2.h]
ude = UserDefinedEquation(
'my ude', my_ude, my_ude_dependents, conns=[Cooling_net_c1, Cooling_net_c2])
ude_2 = UserDefinedEquation(
'my_ude_2', my_ude_2, my_ude_dependents_2, conns=[Cooling_net_c5, Cooling_net_c6])
nw_Cooling_net.add_ude(ude)
nw_Cooling_net.add_ude(ude_2)
Cooling_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5)
Cooling_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5)
Cooling_net_compressor.set_attr(eta_s=0.95)
Cooling_net_c2.set_attr(fluid={"R134a": 1})
            # 6. Parámetros del Consumidor 
Cooling_net_c5.set_attr(T=60, p=3, fluid={"water": 1})
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
Cooling_net_c7.set_attr(T=  6.537e+01 , p=2.5, fluid={"water": 1})
Cooling_net_c8.set_attr(T= 6.037e+01 )
Cooling_net_condenser.set_attr(Q=-15000)
import CoolProp.CoolProp as CP
T_triple = CP.Props1SI("Ttriple", "R134a")        # Triple point temperature (K)
p_triple = CP.Props1SI("ptriple", "R134a")        # Triple point pressure (Pa)
T_critical = CP.Props1SI("T_critical", "R134a")  # Critical temperature (K)
p_critical = CP.Props1SI("p_critical", "R134a")  # Critical pressure (Pa)
h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, "R134a")
T_max_K =  T_critical*0.9
p_high = min(p_critical * 0.9, 30e5) 
h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, "R134a")
nw_Cooling_net._set_p_range([p_triple, p_high])
nw_Cooling_net._set_h_range([h_min,h_max])
nw_Cooling_net.solve('design')


 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 3.57e+06   | 0 %        | 1.28e+01   | 2.42e+05   | 3.87e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 4.92e+05   | 3 %        | 1.58e+01   | 9.10e+03   | 2.08e+06   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 2.63e+05   | 6 %        | 5.10e-01   | 1.74e+01   | 2.64e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 5.36e+04   | 14 %       | 2.32e+00   | 7.36e-05   | 2.18e+03   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 5     | 1.64e+03   | 30 %       | 7.58e-02   | 2.08e-09   | 1.35e-01   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 6     | 5.14e-03   | 92 %       | 2.47e-07   | 4.94e-09   | 3.80e-06   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 7     | 2.64e-04   | 100 %      | 1.37e-09   | 2.47e-09   | 2.56e-04   | 0.00e+00   | 0.00e+00   | 0.0

Invalid value for eff_cold: eff_cold = -0.8095541297407818 below minimum value (0) at component condensador.


In [29]:
nw_Cooling_net.print_results()


##### RESULTS (CycleCloser) #####
+-------------+------------------+-------------------+
|             |   mass_deviation |   fluid_deviation |
|-------------+------------------+-------------------|
| CycleCloser |         0.00e+00 |          0.00e+00 |
+-------------+------------------+-------------------+
##### RESULTS (Compressor) #####
+-----------+----------+----------+-----------+----------+
|           |        P |       pr |        dp |    eta_s |
|-----------+----------+----------+-----------+----------|
| compresor | 5.18e+02 | 1.26e+00 | -3.85e+00 | 9.50e-01 |
+-----------+----------+----------+-----------+----------+
##### RESULTS (Condenser) #####
+-------------+-----------+----------+----------+-----------+----------+----------+----------+----------+------------+----------+------------+----------+------------+
|             |         Q |    ttd_u |    ttd_l |   ttd_min |      pr1 |      pr2 |      dp1 |      dp2 |   zeta1_d4 |    zeta1 |   zeta2_d4 |    zeta2 |   eff_col

In [30]:
nw_Cooling_net.print_equations_with_dependents()

Equations with dependent variables (9 total):
  Eq#  Object       Equation                    Dependent variables
-----  -----------  --------------------------  ---------------------
    0  compresor    eta_s                       h0, h1, p3, p7
    1  condensador  energy_balance_constraints  h1, h2, h4, m5, m6
    2  condensador  Q                           h1, h4, m5
    3  condensador  ttd_u                       p7
    4  condensador  subcooling                  h4, p7
    5  evaporador   energy_balance_constraints  h0, h4, m5, m8
    6  evaporador   ttd_l                       p3, h4
    7  my ude       equation                    h0, p3, h4
    8  my_ude_2     equation                    h2


In [31]:
nw_Cooling_net.print_variables()

Variables after presolving (9 total):
  #  Property    Represents
---  ----------  ---------------------------------
  0  h           2 (h)
  1  h           3 (h)
  2  h           6 (h)
  3  p           0 (p), 1 (p), 2 (p)
  4  h           0 (h), 1 (h), 4 (h)
  5  m           2 (m), 3 (m), 1 (m), 4 (m), 0 (m)
  6  m           5 (m), 6 (m)
  7  p           3 (p), 4 (p)
  8  m           7 (m), 8 (m)


In [32]:
nw_Cooling_net.print_incidence_matrix()

Incidence matrix:
                                        h0    h1    h2    p3    h4    m5    m6    p7    m8
--------------------------------------  ----  ----  ----  ----  ----  ----  ----  ----  ----
compresor.eta_s                         x     x     -     x     -     -     -     x     -
condensador.energy_balance_constraints  -     x     x     -     x     x     x     -     -
condensador.Q                           -     x     -     -     x     x     -     -     -
condensador.ttd_u                       -     -     -     -     -     -     -     x     -
condensador.subcooling                  -     -     -     -     x     -     -     x     -
evaporador.energy_balance_constraints   x     -     -     -     x     x     -     -     x
evaporador.ttd_l                        -     -     -     x     x     -     -     -     -
my ude.equation                         x     -     -     x     x     -     -     -     -
my_ude_2.equation                       -     -     x     -     -     -     - 

In [33]:
nw_Cooling_net.print_presolved_equations()

Presolved equations (22 total):
Object             Equation
-----------------  ----------------------------
5                  T
7                  T
8                  T
CycleCloser        pressure_equality_constraint
CycleCloser        enthalpy_equality_constraint
compresor          mass_flow_constraints
compresor          fluid_constraints
condensador        mass_flow_constraints
condensador        mass_flow_constraints
condensador        fluid_constraints
condensador        fluid_constraints
condensador        pr1
condensador        pr2
evaporador         mass_flow_constraints
evaporador         mass_flow_constraints
evaporador         fluid_constraints
evaporador         fluid_constraints
evaporador         pr1
evaporador         pr2
valvula_expansion  mass_flow_constraints
valvula_expansion  fluid_constraints
valvula_expansion  enthalpy_constraints
